In [ ]:
# Import Modules
import sys
import os
import time
import numpy as np
from RHEMSnow import Preprocess
from RHEMSnow import Model
from RHEMSnow import Output
import scipy.io as sio

In [ ]:
# Names of Forcing Data Files

forcing_files = []
forcing_files.append('Data/Cligen/Initial Sites/wy485055.stm')
forcing_files.append('Data/Cligen/Initial Sites/sd396597.stm')
forcing_files.append('Data/Cligen/Initial Sites/nm293706.stm')
forcing_files.append('Data/Cligen/Initial Sites/nd320995.stm')
forcing_files.append('Data/Cligen/Initial Sites/mt243176.stm')
forcing_files.append('Data/Cligen/Initial Sites/az021001.stm')

validation_file = []

In [ ]:
# Define Model Parameters

# Get default model parameters
nlocs = len(forcing_files)
model_pars = Preprocess.default_model_pars(nlocs)

# For now, set soil parameters from Clapp and Hornberger based on soil type; slope and aspect from excel file
# model_pars['ssat'][:] = [0.451, 0.476, 0.451, 0.451, 0.451, 0.435]
# model_pars['b_soil'][:] = [5.39, 8.52, 5.39, 5.39, 5.39, 4.9]
# model_pars['k_soil'][:] = [600.48, 211.68, 600.48, 600.48, 600.48, 2995.2]

model_pars['Soil'] = ['Loam','Clay loam','Loam','Loam','Loam','Sandy loam']
model_pars = Preprocess.get_soil_pars(model_pars)

model_pars['slope'][:] = [5.14, 22.78, 2.86, 1.72, 4, 3.43]
model_pars['aspect'][:] = [270, 180, 225, 335, 225, 45]

with open('Parameters.csv') as f:
    c = 0
    names = []
    RainThreshs = []
    for line in f:
        if c > 0:
            [name,RainThresh] = line.strip().split(',')
            names.append(name)
            RainThreshs.append(float(RainThresh))
        c = c+1
names = np.array(names)
RainThreshs = np.array(RainThreshs)

model_pars['RainThresh'] = []
for forcing_file in forcing_files:
    loc = names == os.path.basename(forcing_file).replace('.stm','')
    model_pars['RainThresh'].append(RainThreshs[loc][0])
    
# model_pars['RainThresh'][:] = [-100, -100, -100, -100, -100, -100]   # To get rain only simulations


In [ ]:
# Run the model

# Read forcing data
t = time.time()
TS_vec, forcing_data = Preprocess.get_forcing_cligen(forcing_files,model_pars)
print('Elapsed time is ' + str(time.time() - t) + ' seconds')

# Run Rhem-Snow
t = time.time()
model_output = Model.run_model(TS_vec,forcing_data,model_pars)
print('Elapsed time is ' + str(time.time() - t) + ' seconds')

# Disaggregate net water input timeseries
t = time.time()
[TSRainfall,TSMelt] = Model.get_ts_data(forcing_data,model_output,1/288)
print('Elapsed time is ' + str(time.time() - t) + ' seconds')


In [ ]:
# Collect additional data to be outputted

sat = np.ones(model_output['SMC'].shape)
ice = np.ones(model_output['SMC'].shape)
for loc in range(len(model_output['SMC'][0,:])):
    hi = np.percentile(model_output['SMC'][:,loc],99)
    lo = np.percentile(model_output['SMC'][:,loc],1)
    sat[:,loc] = np.maximum(0,np.minimum(1,(model_output['SMC'][:,loc] - lo) / (hi-lo)))
    ice[:,loc] = np.maximum(0,np.minimum(1,(model_output['ice_fraction_soil'][:,loc] - lo) / (hi-lo))) * model_pars['ssat'][loc]
    
TSPrecip = TSMelt + TSRainfall

rainfall = forcing_data['rainfall']
snowfall = forcing_data['snowfall']
rain_on_snow = model_output['rain_on_snow']
rain_off_snow = rainfall - rain_on_snow
melt = model_output['melt']
net_water_input = rain_off_snow + melt;

# Find Maximum Intensity

MaxIntensity = np.zeros(model_output['swe'].shape) * np.nan

print('Finding Maximum Intensity')
for i in range(len(TS_vec)):
    precip = TSPrecip[i*288+1:(i+1)*288,:]
    precip_30 = np.zeros([48, len(precip[0,:])]) * np.nan
    
    for j in range(48):
        precip_30[j,:] = np.sum(precip[j*6+1:(j+1)*6,:],axis=0)

    MaxIntensity[i,:] = np.max(precip_30,axis=0)


In [ ]:
# Output the 5 minute model data

ids = ['wy485055', 'sd396597', 'nm293706', 'nd320995', 'mt243176', 'az021001']
OutDir = 'Output/CligenSites'

t = time.time()
print('Writing output files')
Output.write_ts_output(OutDir + '/' + ids[0] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[0]], TSPrecip[:, 0], net_water_input[:, 0], sat[:, 0], ice[:, 0])
Output.write_ts_output(OutDir + '/' + ids[1] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[1]], TSPrecip[:, 1], net_water_input[:, 1], sat[:, 1], ice[:, 1])
Output.write_ts_output(OutDir + '/' + ids[2] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[2]], TSPrecip[:, 2], net_water_input[:, 2], sat[:, 2], ice[:, 2])
Output.write_ts_output(OutDir + '/' + ids[3] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[3]], TSPrecip[:, 3], net_water_input[:, 3], sat[:, 3], ice[:, 3])
Output.write_ts_output(OutDir + '/' + ids[4] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[4]], TSPrecip[:, 4], net_water_input[:, 4], sat[:, 4], ice[:, 4])
Output.write_ts_output(OutDir + '/' + ids[5] + '_output', forcing_data['year'][:,0], forcing_data['mon'][:,0], forcing_data['day'][:,0], 1/288, [ids[5]], TSPrecip[:, 5], net_water_input[:, 5], sat[:, 5], ice[:, 5])

# Output.write_daily_mat(OutDir + '/Output.mat', model_output, forcing_data, model_pars, TS_vec, TSPrecip)
print('Elapsed time is ' + str(time.time() - t) + ' seconds')


In [ ]:
# Output additional model data

OutDir = 'Output/CligenSites'

# Output Mat File

a = {}
a['forcing_data'] = forcing_data
a['model_output'] = model_output
a['TSRainfall'] = TSRainfall
a['TSMelt'] = TSMelt
a['MaxIntensity'] = MaxIntensity

fname = OutDir + '/model_data.mat'
print('Saving ' + fname)
sio.savemat(fname,a)

# Output Table 

for i in range(len(forcing_files)):
    year = forcing_data['year'][:,i]
    mon = forcing_data['mon'][:,i]
    day = forcing_data['day'][:,i]
    rainfall = forcing_data['rainfall'][:,i]
    snowfall = forcing_data['snowfall'][:,i]
    rain_on_snow = model_output['rain_on_snow'][:,i]
    rain_off_snow = rainfall - rain_on_snow
    swe = model_output['swe'][:,i]
    snowpack_sublimation = model_output['snowpack_sublimation'][:,i]
    melt = model_output['melt'][:,i]
    ice_ = ice[:,i]
    sat_ = sat[:,i]
    net_water_input = rain_off_snow + melt;
    OutTable = np.array([year, mon, day, rain_off_snow, rain_on_snow, snowfall, swe, snowpack_sublimation, melt, net_water_input, ice_, sat_, MaxIntensity[:,i]]).T
    
    fname = OutDir + '/' + ids[i] + '_table.csv'
    print('Saving ' + fname)
    header = 'year,month,day,Rainfall Off Snow (mm),Rainfall On Snow (mm),Snowfall (mm),End of Day SWE (mm),Sublimation (mm),Snowmelt (mm),Net Water Input (mm),Ice Fraction,Saturation Fraction,Intensity (mm/30min)'
    np.savetxt(fname, OutTable, delimiter=',', header=header, comments='')
